## I. Setup Software and Some Libraries

We first need to set the working environment and the path to the dataset.

In [1]:
import sys

SOFTWARE_DIR = '/mnt/data/spine_bilal/spine/' # Change this path to your software install
DATA_DIR = '/mnt/data/polaris/2025_02_17/inference/mc_handscan/' # Change this path if you are not on SDF (see main README)

# SOFTWARE_DIR = '/home/bilal/spine_bilal/spine/'  # Adjust if needed
# DATA_DIR = '/home/bilal/inference/mc_handscan/'

# Set software directory
sys.path.append(SOFTWARE_DIR)

import numpy as np
import math
import pandas as pd
from collections import OrderedDict

from scipy.spatial.distance import cdist

Now pass the analysis configuration.

In [113]:
import yaml
from spine.driver import Driver


DATA_PATH = DATA_DIR + 'MiniRun6.1_1E19_RHC.spine.0000040.MLRECO_SPINE.hdf5'

cfg = '''
# Load HDF5 files
io:
  reader:
    name: hdf5
    file_keys: DATA_PATH
    skip_unknown_attrs: true
# Build reconstruction output representations
build:
  mode: both
  units: cm
  fragments: false
  particles: true
  interactions: true
'''.replace('DATA_PATH', DATA_PATH)

cfg = yaml.safe_load(cfg)
driver = Driver(cfg)


 ██████████   ██████████    ███   ███       ██   ███████████
███        █  ██       ███   █    █████     ██   ██         
  ████████    ██       ███  ███   ██  ████  ██   ██████████ 
█        ███  ██████████     █    ██     █████   ██         
 ██████████   ██            ███   ██       ███   ███████████

Release version: 0.2.2

$CUDA_VISIBLE_DEVICES=

Configuration processed at: Linux bd9fe1b7ae37 5.15.167.4-microsoft-standard-WSL2 #1 SMP Tue Nov 5 00:21:55 UTC 2024 x86_64 x86_64 x86_64 GNU/Linux

base: {seed: 1742341660}
io:
  reader: {name: hdf5, file_keys: /mnt/data/polaris/2025_02_17/inference/mc_handscan/MiniRun6.1_1E19_RHC.spine.0000040.MLRECO_SPINE.hdf5,
    skip_unknown_attrs: true}
build: {mode: both, units: cm, fragments: false, particles: true, interactions: true}

Will load 1 file(s):
  - /mnt/data/polaris/2025_02_17/inference/mc_handscan/MiniRun6.1_1E19_RHC.spine.0000040.MLRECO_SPINE.hdf5

Total number of entries in the file(s): 179

Total number of entries selected: 179


Map ND-LAr events to their corresponding `entry` numbers in SPINE files using the `run_info` field.

In [114]:
import sys
import numpy as np
import h5py
from tabulate import tabulate  # Install using: pip install tabulate
from spine.driver import Driver
from spine.utils.globals import PID_LABELS  # Ensure PID_LABELS maps numbers to particle names

print('Number of entries in the loader:', len(driver))

# Open the HDF5 file to read event details
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    total_entries = len(run_info)

    # Initialize results storage
    all_table_data = []

    for ENTRY in range(total_entries):
        # Get the target event from run_info
        try:
            run, subrun, target_event = run_info[ENTRY]  # Extract event number
        except IndexError:
            print(f"Skipping entry {ENTRY} due to missing run_info.")
            continue

        # Process data for the current entry
        try:
            data = driver.process(entry=ENTRY)
            reco_particles = data['reco_particles']
            truth_particles = data['truth_particles']
        except Exception as e:
            print(f"Skipping entry {ENTRY} due to error: {e}")
            continue  # Skip problematic entries

        # Count reco and truth particles per interaction
        reco_interaction_particle_counts = {}
        truth_interaction_particle_counts = {}

        # Initialize particle type counts per interaction
        reco_pid_counts = {}
        truth_pid_counts = {}

        # Process reco particles
        for particle in reco_particles:
            interaction_id = particle.interaction_id
            pid = particle.pid

            # Update reco interaction count
            reco_interaction_particle_counts[interaction_id] = reco_interaction_particle_counts.get(interaction_id, 0) + 1
            
            # Count muons, pions, and protons per interaction
            if interaction_id not in reco_pid_counts:
                reco_pid_counts[interaction_id] = {2: 0, 3: 0, 4: 0}  # Muon, Pion, Proton

            if pid in reco_pid_counts[interaction_id]:
                reco_pid_counts[interaction_id][pid] += 1

        # Process truth particles
        for particle in truth_particles:
            interaction_id = particle.interaction_id
            pid = particle.pid

            # Update truth interaction count
            truth_interaction_particle_counts[interaction_id] = truth_interaction_particle_counts.get(interaction_id, 0) + 1

            # Count muons, pions, and protons per interaction
            if interaction_id not in truth_pid_counts:
                truth_pid_counts[interaction_id] = {2: 0, 3: 0, 4: 0}  # Muon, Pion, Proton

            if pid in truth_pid_counts[interaction_id]:
                truth_pid_counts[interaction_id][pid] += 1

        # Collect interaction IDs present in either reco or truth
        interaction_ids = sorted(set(reco_interaction_particle_counts.keys()).union(truth_interaction_particle_counts.keys()))

        # Store table data for this entry
        for interaction_id in interaction_ids:
            reco_count = reco_interaction_particle_counts.get(interaction_id, 0)
            truth_count = truth_interaction_particle_counts.get(interaction_id, 0)

            # Get muon, pion, proton counts (default to 0 if not found)
            reco_muons = reco_pid_counts.get(interaction_id, {}).get(2, 0)
            reco_pions = reco_pid_counts.get(interaction_id, {}).get(3, 0)
            reco_protons = reco_pid_counts.get(interaction_id, {}).get(4, 0)

            truth_muons = truth_pid_counts.get(interaction_id, {}).get(2, 0)
            truth_pions = truth_pid_counts.get(interaction_id, {}).get(3, 0)
            truth_protons = truth_pid_counts.get(interaction_id, {}).get(4, 0)

            # Format counts as "Reco/Truth"
            muon_str = f"{reco_muons}/{truth_muons}"
            pion_str = f"{reco_pions}/{truth_pions}"
            proton_str = f"{reco_protons}/{truth_protons}"

            # Append row to table
            all_table_data.append([
                ENTRY, target_event, interaction_id, 
                reco_count, truth_count, 
                muon_str, pion_str, proton_str
            ])

# Print final tabulated output
print("\nSummary of Particles per Interaction ID for All Entries:")
print(tabulate(all_table_data, headers=[
    "Entry", "Event ID", "Interaction ID", 
    "Reco Particles", "Truth Particles", 
    "Muons (R/T)", "Pions (R/T)", "Protons (R/T)"
], tablefmt="pretty"))


Number of entries in the loader: 179

Summary of Particles per Interaction ID for All Entries:
+-------+----------+----------------+----------------+-----------------+-------------+-------------+---------------+
| Entry | Event ID | Interaction ID | Reco Particles | Truth Particles | Muons (R/T) | Pions (R/T) | Protons (R/T) |
+-------+----------+----------------+----------------+-----------------+-------------+-------------+---------------+
|   2   |    3     |       0        |       9        |       13        |     3/1     |     0/2     |      3/6      |
|   2   |    3     |       1        |       3        |        3        |     1/1     |     0/0     |      0/0      |
|   2   |    3     |       2        |       3        |        4        |     1/1     |     0/0     |      0/0      |
|   3   |    4     |       0        |       13       |       10        |     1/1     |     1/1     |      2/4      |
|   3   |    4     |       1        |       1        |        1        |     0/0     |

In [115]:
import sys
import numpy as np
import h5py
import os
import plotly.io as pio
from tabulate import tabulate  # Install using: pip install tabulate
from spine.driver import Driver
from spine.utils.globals import PID_LABELS  # Ensure PID_LABELS maps numbers to particle names
from spine.vis.out import Drawer

# Define spatial limits
distance_from_wall = 5.0
minX = -63.931 + distance_from_wall
maxX = +63.931 - distance_from_wall
minY = -62.076 + distance_from_wall
maxY = +62.076 - distance_from_wall
minZ = -64.538 + distance_from_wall
maxZ = +64.538 - distance_from_wall

# Open the HDF5 file to read event details
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    total_entries = len(run_info)

    for ENTRY in range(total_entries):
        # Get the target event from run_info
        try:
            run, subrun, target_event = run_info[ENTRY]  # Extract event number
        except IndexError:
            print(f"Skipping entry {ENTRY} due to missing run_info.")
            continue

        # Process data for the current entry
        try:
            data = driver.process(entry=ENTRY)
            reco_particles = data['reco_particles']
            truth_particles = data['truth_particles']
        except Exception as e:
            print(f"Skipping entry {ENTRY} due to error: {e}")
            continue  # Skip problematic entries

        # Extract all unique interaction IDs
        interaction_ids = sorted(set(particle.interaction_id for particle in truth_particles))
        print(f"\nEntry {ENTRY} contains {len(interaction_ids)} interactions: {interaction_ids}")

        # Collect all particles for visualization
        all_reco_particles = []
        all_truth_particles = []
        matched_truth_particle_ids = set()
        selected_interaction_ids = set()  # Stores interactions that pass the selection criteria

        for TARGET_INTERACTION_ID in interaction_ids:
            # Get particles for the current interaction
            reco_particles = [p for p in data['reco_particles'] if p.interaction_id == TARGET_INTERACTION_ID]
            truth_particles = [p for p in data['truth_particles'] if p.interaction_id == TARGET_INTERACTION_ID]
            
            # Apply selection criteria for truth particles
            passes_selection = False
            for particle in truth_particles:
                if particle.is_primary:
                    x, y, z = particle.start_point  # Extract start point coordinates
                    if minX < x < maxX and minY < y < maxY and minZ < z < maxZ:
                        passes_selection = True
                        break  # If one truth particle passes, we keep the entire interaction
            
            if not passes_selection:
                continue  # Skip interactions that don't satisfy the spatial constraints
            
            # If it passes, add its particles for visualization
            selected_interaction_ids.add(TARGET_INTERACTION_ID)
            all_reco_particles.extend(reco_particles)
            all_truth_particles.extend(truth_particles)

            # Match truth particle IDs with reco
            for reco_particle in all_reco_particles:
                for reco_p, truth_p in data['particle_matches_r2t']:
                    if reco_p and truth_p:  # Ensure neither reco_p nor truth_p is None
                        if reco_p.id == reco_particle.id:
                            matched_truth_particle_ids.add(truth_p.id)

        # If no interactions pass the selection criteria, skip visualization
        if not selected_interaction_ids:
            print(f"Skipping Entry {ENTRY}: No valid interactions passed spatial selection.")
            continue

        # Filter truth interactions to include only selected ones
        filtered_truth_interactions = [
            interaction for interaction in data['truth_interactions']
            if interaction.id in selected_interaction_ids
        ]
        
        # Create a Drawer instance
        drawer = Drawer(data, draw_mode='both', detector='2x2', split_scene=True)
        
        # Pre-filter the data for visualization
        data['reco_particles'] = all_reco_particles
        data['truth_particles'] = [p for p in all_truth_particles if p.id in matched_truth_particle_ids]
        data['truth_interactions'] = filtered_truth_interactions
        
        # Generate the visualization
        fig = drawer.get(
            obj_type='particles',
            attr='pid',
            draw_end_points=True,
            draw_vertices=True,
            synchronize=True,
            titles=[f"Reconstructed Event", f"Truth Event"],
            split_traces=True
        )
        
        # Adjust the figure layout for better visibility
        fig.update_layout(height=800, width=1000)
        
        # Extract the file number (e.g., '0000000') from DATA_PATH
        file_number = os.path.basename(DATA_PATH).split('.')[3]
        
        # Construct dynamic filenames
        html_filename = f"MR6p1_{file_number}_entry_{ENTRY}_all_interactions.html"
        png_filename = f"MR6p1_{file_number}_entry_{ENTRY}_all_interactions.png"
        
        # Save visualization as HTML
        pio.write_html(fig, file=html_filename)
        print(f"\nVisualization saved as HTML: {html_filename}")
        
        # Save visualization as PNG
        pio.write_image(fig, file=png_filename, format='png', width=1000, height=800)
        print(f"Visualization saved as PNG: {png_filename}")


Entry 0 contains 0 interactions: []
Skipping Entry 0: No valid interactions passed spatial selection.

Entry 1 contains 0 interactions: []
Skipping Entry 1: No valid interactions passed spatial selection.

Entry 2 contains 3 interactions: [0, 1, 2]

Visualization saved as HTML: MR6p1_0000040_entry_2_all_interactions.html
Visualization saved as PNG: MR6p1_0000040_entry_2_all_interactions.png

Entry 3 contains 2 interactions: [0, 1]

Visualization saved as HTML: MR6p1_0000040_entry_3_all_interactions.html
Visualization saved as PNG: MR6p1_0000040_entry_3_all_interactions.png

Entry 4 contains 1 interactions: [0]
Skipping Entry 4: No valid interactions passed spatial selection.

Entry 5 contains 4 interactions: [0, 1, 2, 3]

Visualization saved as HTML: MR6p1_0000040_entry_5_all_interactions.html
Visualization saved as PNG: MR6p1_0000040_entry_5_all_interactions.png

Entry 6 contains 0 interactions: []
Skipping Entry 6: No valid interactions passed spatial selection.

Entry 7 contains 3 